# Hyena — a toy-scale build of an implicit long-convolution operator

A minimal implementation of the **Hyena operator**, from Poli et al.,
*"Hyena Hierarchy: Towards Larger Convolutional Language Models"* (2023) —
a way to replace attention entirely with long convolutions, generated
on-the-fly by a small neural network instead of stored as raw weights.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

Attention computes, for every pair of tokens, how much they should interact
— which is powerful, but scales quadratically with sequence length. Hyena
asks: what if instead we used a **convolution** — cheaper, and with a clear
inductive bias for "nearby things interact more" — but made it long enough
to reach across the *entire* sequence, and let the model *control* the
convolution with gates, the way attention lets the model control what to
attend to?

Two ingredients make this work:

- **Implicit long convolutions.** A convolution filter that reaches across
  the whole sequence would normally mean storing a huge number of weights
  (one per position, per channel). Instead, Hyena **generates** the filter
  from a small MLP: feed the MLP a position (as a sinusoidal embedding,
  like a positional encoding), and it outputs what the filter's value
  should be at that position. The filter is computed on the fly, for
  whatever sequence length is needed, from a tiny number of MLP weights —
  not stored directly.
- **Data-controlled gating.** A single long convolution is still just a
  fixed linear operation, no matter how it's parameterized — it doesn't
  look at the data to decide what to do. Hyena interleaves convolutions with
  **elementwise multiplication by a projection of the input itself**, which
  is what actually makes the operator data-dependent, the way attention's
  weights depend on the actual tokens being attended to.

## 2. The recurrence, order by order

A Hyena operator of order `N` projects the input into `N+1` branches, then
alternates "long convolution" and "gate by another branch," `N` times:

```
x0, x1, ..., x_{N-1}, v = split(project(input))
z = v
for n in 0 .. N-1:
    z = long_conv_n(z) * x_n
output = project(z)
```

This notebook builds the **order-2** case (so 3 branches: `x0`, `x1`, `v`):

```
z = v
z = long_conv_0(z) * x0
z = long_conv_1(z) * x1
output = project(z)
```

Each `long_conv_n` uses its *own* implicit filter (its own small MLP), and
because the filter only has non-zero values for lags `0..t` (it's
constructed to be causal — see below), the whole operation only ever looks
at the past.

## 3. Making the long convolution both causal and fast

Two details matter for a long convolution to actually work:

- **Causality**: filter position `k` should only ever combine with `x_{t-k}`
  for `k >= 0` — the filter is naturally causal because it's only defined
  and only ever used for non-negative lags relative to the current
  position.
- **An exponential decay envelope**: the raw MLP-produced filter is
  multiplied by a decaying envelope (`exp(-decay * t)`), so distant lags are
  naturally down-weighted rather than left free to have arbitrary
  magnitude — this is a stability trick the paper also uses, since an
  unconstrained implicit filter can otherwise be numerically unstable
  during training.
- **Speed via FFT**: computing a convolution with a filter as long as the
  whole sequence naively costs `O(T^2)`, same as attention. Using the
  convolution theorem (`conv(x, f) = ifft(fft(x) * fft(f))`), it can be
  computed in `O(T log T)` instead — this is Hyena's actual efficiency
  argument over attention at long sequence lengths.

> **Simplification used here:** the paper's Hyena operator typically
> includes short convolutions on the projected branches too (mirroring the
> short conv used ahead of KDA/Mamba's recurrences elsewhere in this repo),
> along with more careful filter regularization. This notebook keeps just
> the two essential ingredients — implicit long convolutions and
> data-controlled gating — to keep the core idea legible.

In [ ]:
class HyenaFilter(nn.Module):
    '''Produces a long implicit convolution filter from a small MLP over
    sinusoidal positional embeddings, instead of storing T filter weights directly.'''
    def __init__(self, d_channels, d_pos=8, hidden=16):
        super().__init__()
        freqs = torch.linspace(1, 64, d_pos // 2)
        self.register_buffer('freqs', freqs)
        self.mlp = nn.Sequential(nn.Linear(d_pos, hidden), nn.GELU(), nn.Linear(hidden, d_channels))
        self.decay = nn.Parameter(torch.ones(d_channels) * 0.1)   # controls how fast far positions fade

    def forward(self, L):
        t = torch.arange(L, device=self.freqs.device).float()
        ang = t.view(-1, 1) * self.freqs.view(1, -1)
        pos_emb = torch.cat([torch.sin(ang), torch.cos(ang)], dim=-1)     # L, d_pos
        h = self.mlp(pos_emb)                                              # L, d_channels -- the raw filter
        env = torch.exp(-F.softplus(self.decay).view(1, -1) * t.view(-1, 1))   # causal decay envelope
        return (h * env).transpose(0, 1)                                    # d_channels, L

def causal_fft_conv(x, filt):
    # x: B,D,T   filt: D,T  ->  causal convolution via FFT (conv theorem)
    B, D, T = x.shape
    n = 2 * T
    xf = torch.fft.rfft(x, n=n)
    ff = torch.fft.rfft(filt, n=n)
    return torch.fft.irfft(xf * ff, n=n)[..., :T]

In [ ]:
class HyenaOperator(nn.Module):
    '''Order-2 Hyena: project to 3 branches (x0, x1, v), then
    z <- long_conv_0(v)  * x0
    z <- long_conv_1(z) * x1
    -- each step is a data-controlled gate multiplying an implicit long convolution.'''
    def __init__(self, d_model=64, order=2):
        super().__init__()
        self.order = order
        self.in_proj = nn.Linear(d_model, d_model * (order + 1), bias=True)
        self.short_conv = nn.Conv1d(d_model * (order + 1), d_model * (order + 1), 3,
                                     groups=d_model * (order + 1), padding=2)
        self.filters = nn.ModuleList([HyenaFilter(d_model) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.d_model = d_model

    def forward(self, x):
        B, T, D = x.shape
        z = self.in_proj(x).transpose(1, 2)                 # B, D*(order+1), T
        z = self.short_conv(z)[..., :T]                       # small local mixing before the long convs
        branches = z.chunk(self.order + 1, dim=1)             # each B,D,T
        v, xs = branches[-1], branches[:-1]
        out = v
        for n in range(self.order):
            filt = self.filters[n](T)                          # D,T -- generated fresh for this sequence length
            out = causal_fft_conv(out, filt) * xs[n]            # long conv, then data-controlled gate
        return self.out_proj(out.transpose(1, 2))              # B,T,D

## 4. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([HyenaOperator(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through the Hyena operator once it's wired into a real model. So the rest of this
notebook:

1. wraps the Hyena operator into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Visualize a learned filter.** Call one of the trained `HyenaFilter`
  modules directly and plot its output over positions — you should see it
  develop structure (rather than staying close to its random initialization)
  once training has taught it something about the periodic pattern.
- **Try a higher order** (`order=3` or more) — each additional order adds
  another gate-then-convolve step, giving the operator more capacity to
  build up higher-order interactions between positions, at the cost of more
  branches to project into.
- **Compare wall-clock scaling** against the attention-based notebooks in
  this repo (`../mla`, `../nsa`) as you increase sequence length — Hyena's
  `O(T log T)` long convolution should scale more gently than the `O(T^2)`
  cost of dense attention.

Reference: Poli, Massaroli, Nguyen, Fu, Dao, Baccus, Bengio, Ermon, Ré,
*"Hyena Hierarchy: Towards Larger Convolutional Language Models,"* 2023.